# Modelling AusRAP: a road-safety dataset made of codes

> **Attribution.** AusRAP data © State of Victoria (Department of Transport and
> Planning), licensed [CC-BY 4.0](https://creativecommons.org/licenses/by/4.0/).
> Source: <https://opendata.transport.vic.gov.au/dataset/australian-road-assessment-program-ausrap>

The second hands-on notebook on schemas. [Notebook 5](5-schema-models.ipynb) taught
the pieces (fields, types, enums, rules) on a toy `RoadSafetyRating`. Here you use
them on a real dataset: **AusRAP Victoria**, the star ratings that Victoria's
Department of Transport and Planning publishes for its road network, produced with
the [iRAP](https://irap.org/) methodology.

The notebook ends in a schema: a Pydantic model for one AusRAP record, with every
coded column as a documented enum, checked against every row of the real file.
It stops there. Turning a schema into an Overture extension, and matching AusRAP to
Overture's roads, are separate jobs.

Sections:

1. Get the data and see what is in it.
2. Codes or labels? Look before designing.
3. What a record means: 100 m bins, worst-within-100 m.
4. Enums: one per coded column, each value documented.
5. The record: structs, a rule, and a geometry.
6. Check the schema against the data.
7. Emit the schema.

Some cells are *meant* to fail. A failure prints a `ValidationError`; the last lines
name the field and the rule it broke.

In [ ]:
%xmode Minimal
import urllib.request
from pathlib import Path

import duckdb

# Resolve paths from the notebook's location, so it works whether Jupyter was
# launched from the repo root or from notebooks/.
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
DATA = REPO_ROOT / "data"
RESULTS = REPO_ROOT / "results"

con = duckdb.connect()

## 1. Get the data

The full state file is about 80 MB, so it is not in the repo. The cell below
downloads it once into `data/` (which is gitignored) and reuses it afterwards.

In [ ]:
AUSRAP_URL = (
    "https://opendata.transport.vic.gov.au/dataset/"
    "da201a82-372b-4f23-8e88-9819a6672bd0/resource/"
    "4148542e-6e7b-476f-bcb2-de94114bc40a/download/ausrap.csv"
)
AUSRAP = DATA / "ausrap.csv"

if not AUSRAP.exists():
    DATA.mkdir(exist_ok=True)
    print("Downloading AusRAP Victoria (~80 MB, CC-BY 4.0, one time)...")
    urllib.request.urlretrieve(AUSRAP_URL, AUSRAP)

con.execute(
    f"CREATE TABLE ausrap AS SELECT * FROM read_csv_auto('{AUSRAP.as_posix()}', sample_size=-1)"
)
con.sql("SELECT count(*) AS rows, count(DISTINCT \"Location ID\") AS location_ids FROM ausrap")

63 columns, one row per 100 m of one carriageway. `Location ID` is unique per row.
First, which columns hold what kind of value?

In [ ]:
con.sql("""
    SELECT column_type, count(*) AS columns, list(column_name ORDER BY column_name)[:4] AS first_four
    FROM (DESCRIBE ausrap) GROUP BY column_type ORDER BY columns DESC
""")

## 2. Codes or labels?

Most columns are `BIGINT`: the coded ones describe road features (median type, speed
limit, roadside hazard), yet hold integers. Count the values in two of them.

In [ ]:
con.sql('SELECT "Median type" AS median_type, count(*) AS rows FROM ausrap GROUP BY 1 ORDER BY 1')

In [ ]:
con.sql('SELECT "Speed limit" AS speed_limit, count(*) AS rows FROM ausrap GROUP BY 1 ORDER BY 1')

These are **iRAP numeric codes**, not labels and not measurements. The speed limit
column holds `1, 3, 5, 7, ...`, not kilometres per hour. A value means nothing until
you look it up, and the lookup table is the part a schema has to carry.

**Where the meanings come from.** The [iRAP Coding
Manual](https://irap.org/specifications/) defines each attribute's codes. This
notebook uses Version 5.3, Drive on Left edition (2022), section 3 for the
definitions and section 4 (Quick Coding Guide) for the tables. Two caveats on that
choice:

- The Austroads copy of the *Star Rating and Investment Plan Manual* v3.10 sits
  behind a bot check and could not be fetched, so the codes here come from the
  Coding Manual alone. The two manuals describe the same coding scheme, but this
  notebook did not compare them.
- The dataset does not state which edition it was coded under (the survey dates
  are 2022 and 2023), and the manual's changelog shows code sets have changed:
  new pedestrian crossing categories and school zone attributes in 2013, and
  corrections for differential speeds and refuge islands in 2020. Section 6 checks
  that every code in the file is defined by the 5.3 tables, which is evidence the
  editions agree here, not proof.

**Which meanings were checked against the data.** Reading a manual proves what the
manual says. Two meanings can be tested against AusRAP itself:

1. Speed limit codes. Rows whose `Landmark` is a speed-limit sign ("Speed Limit 60
   km/hr") give a posted limit to compare with the code. For each code, the most
   common sign:

In [ ]:
con.sql("""
    WITH signs AS (
        SELECT "Speed limit" AS code,
               regexp_extract(Landmark, 'Speed Limit (\\d+)', 1)::INT AS sign_kmh,
               count(*) AS rows
        FROM ausrap WHERE Landmark LIKE 'Speed Limit%'
        GROUP BY 1, 2
    )
    SELECT code, sign_kmh, rows FROM signs
    QUALIFY rows = max(rows) OVER (PARTITION BY code)
    ORDER BY code
""")

Every code with more than a handful of signed rows lines up with the manual's
table: 3 is 40 km/h, 5 is 50, 7 is 60, 9 is 70, 11 is 80, 13 is 90, 15 is 100, 17
is 110. Code 1 ("under 30") has three signed rows. Signs and codes disagree on
some rows, as expected: a code summarises a whole 100 m, a landmark is one point.

2. Roadworks. The manual says "star ratings will not be generated" for major road
works (code 3). The data agrees:

In [ ]:
con.sql("""
    SELECT Roadworks AS roadworks_code,
           count(*) AS rows,
           count(*) FILTER (WHERE "Vehicle Star Rating Smoothed" IS NULL) AS without_star_rating
    FROM ausrap GROUP BY 1 ORDER BY 1
""")

Every other value's meaning rests on the manual alone. That includes the
`Vehicle SRS ...` score columns: the Coding Manual does not define scores (the Star
Rating manual does), so their descriptions in the schema below come from the column
names and are marked as such.

## 3. What a record means

Three facts about iRAP data decide the schema's shape:

- **One row is a 100 m bin.** Every row has `Length` 0.1 (km), a start coordinate
  (`Latitude`, `Longitude`) and an end coordinate (`End Latitude`, `End Longitude`).
  The geometry is the straight line between them: a `LineString`.
- **Each attribute is the worst case within the 100 m.** The manual: "if a 100 m
  segment includes a cliff for 50 m and a safety barrier for the next 50 m, record
  the cliff." A value describes a bin, not a place. Its enum values are listed
  from highest to lowest risk for this reason.
- **Each carriageway is its own row set.** A divided road is surveyed once per
  direction (`Carriageway` 1 and 2); an undivided road once (code 3). Sides are
  relative to the direction of survey: "driver-side" and "passenger-side".

There is no persistent segment ID. `Location ID` identifies a row in this file; a
re-survey of the same road gets new IDs and bins that no longer line up.

The target is one record per bin:

```
geometry           LineString, start to end of the bin
location_id        this file's ID for the bin
carriageway        divided A, divided B, undivided
star_rating        { car, car_raw }                  1 to 5
srs                { run_off..., head_on..., total } risk scores
roadworks          none | minor | major
speed_limit        a code, documented as km/h
roadside_*         { distance, object } for each side of the road
...                and every other coded column, as an enum
```

## 4. Enums: one per coded column

A coded column becomes a `DocumentedEnum` whose values are the codes the data holds
and whose docs say what each means. Notebook 5 wrote one by hand:

```python
class RoadType(str, DocumentedEnum):
    MOTORWAY = ("motorway", "Divided highway with controlled access.")
```

Here the base is `int`, because the data holds integers. Two decisions, both
visible in the first enum:

- **Keep the codes as the values.** The source's integers validate directly, and
  nothing is lost in translation. The cost is that `3` is opaque in JSON until
  someone reads the docs. The alternative, string values (`"narrow"`) with a
  code-to-string step at ingest, gives readable output and a second mapping to
  maintain. This notebook keeps the codes and leaves the other design as a **Try
  it** at the end.
- **List every value the manual defines, not only the ones this file uses.** A schema
  for AusRAP describes iRAP coding; the 2023 Victorian file is one sample of it.
  (Section 6 reports which defined values this file never uses.)

In [ ]:
from overture.schema.system.doc import DocumentedEnum


class Carriageway(int, DocumentedEnum):
    """Which carriageway a row describes."""

    DIVIDED_A = (1, "Divided carriageway, one direction of travel.")
    DIVIDED_B = (2, "Divided carriageway, the opposite direction to A.")
    UNDIVIDED = (3, "Undivided carriageway, two-way or one-way.")
    MOTORCYCLE_A = (4, "Segregated motorcycle path alongside carriageway A.")
    MOTORCYCLE_B = (5, "Segregated motorcycle path alongside carriageway B.")


for member in Carriageway:
    print(f"{member.value:3} {member.name:14} {member.__doc__}")

Many columns are the same two-value question (a feature is absent or present), and
two are the same two-value judgement (poor or adequate). One enum each, reused.

In [ ]:
class Presence(int, DocumentedEnum):
    """Whether a feature is present along the whole 100 m."""

    NOT_PRESENT = (1, "The feature is absent, or not sufficient to count.")
    PRESENT = (2, "The feature is present.")


class Adequacy(int, DocumentedEnum):
    """A two-level judgement of a feature's standard."""

    ADEQUATE = (1, "Meets the standard.")
    POOR = (2, "Generally absent, in poor condition, or ineffective.")

**Area type, land use, and speed.** Land use is recorded for each side of the road
and drives the assumed pedestrian activity. The manual lists its codes from highest
to lowest risk, which is why the numbers look shuffled: that order is part of the
meaning ("if in doubt between two land uses, select the one that appears first").

In [ ]:
class AreaType(int, DocumentedEnum):
    """Whether the surroundings are developed."""

    RURAL = (1, "Outside an urban area, town or village.")
    URBAN = (2, "Development spanning at least 400 m along the road.")


class LandUse(int, DocumentedEnum):
    """Roadside development, a proxy for pedestrian activity. Listed highest to lowest risk."""

    EDUCATIONAL = (6, "Schools, colleges, universities, hospitals or arenas that generate high pedestrian flows.")
    COMMERCIAL = (4, "Shops, other commercial activity, parks and recreation, or evident high pedestrian activity.")
    INDUSTRIAL = (7, "Industrial units, factories or manufacturing sites.")
    RESIDENTIAL = (3, "Residential housing.")
    FARMING = (2, "Farming or agricultural activity.")
    UNDEVELOPED = (1, "Open land with no shops, housing, industry or farming, or no adjacent land.")
    NOT_RECORDED = (5, "Not recorded. Listed in the manual's quick coding guide only.")


class SpeedLimit(int, DocumentedEnum):
    """The posted speed limit, as an iRAP band. The values are codes, not km/h."""

    UNDER_30_KMH = (1, "Under 30 km/h.")
    KMH_40 = (3, "40 km/h.")
    KMH_50 = (5, "50 km/h.")
    KMH_60 = (7, "60 km/h.")
    KMH_70 = (9, "70 km/h.")
    KMH_80 = (11, "80 km/h.")
    KMH_90 = (13, "90 km/h.")
    KMH_100 = (15, "100 km/h.")
    KMH_110 = (17, "110 km/h.")
    KMH_120 = (19, "120 km/h.")
    KMH_130 = (21, "130 km/h.")
    KMH_140 = (23, "140 km/h.")
    KMH_150_PLUS = (25, "150 km/h or more.")

The manual also defines mph codes (31 to 45) for drive-on-right countries. They are
left out: Victoria posts km/h, and a schema for it should not accept `45` as a
"90 mph or more" speed limit by accident.

**Median type, shoulders, and lanes.** The median code decides whether a road is
divided: barriers and physical medians apply to divided carriageways only; markings
and turning lanes to undivided ones.

In [ ]:
class MedianType(int, DocumentedEnum):
    """What separates opposing traffic."""

    CENTRE_LINE = (11, "Undivided. Single or double centre line, or a marked strip under 0.3 m wide.")
    WIDE_CENTRE_LINE = (14, "Undivided. Centre line or marked strip 0.3 m to 1 m wide.")
    CENTRAL_HATCHING = (10, "Undivided. Marked central strip over 1 m wide.")
    TURNING_LANE = (8, "Undivided. Continuous central lane designated for turning.")
    FLEXIBLE_POSTS = (9, "Undivided. Flexible posts or bollards that are not a continuous physical divider.")
    PHYSICAL_UNDER_1M = (7, "Divided. Defined median of different material, under 1 m wide.")
    PHYSICAL_1_TO_5M = (6, "Divided. Defined median 1 m to under 5 m wide.")
    PHYSICAL_5_TO_10M = (5, "Divided. Defined median 5 m to under 10 m wide.")
    BARRIER_CONCRETE = (2, "Divided. Concrete safety barrier meeting international standards.")
    BARRIER_METAL = (1, "Divided. Metal safety barrier meeting international standards.")
    BARRIER_MOTORCYCLE_FRIENDLY = (12, "Divided. Safety barrier designed to minimise impact to motorcyclists.")
    BARRIER_WIRE_ROPE = (15, "Divided. Safety barrier of tensioned wires meeting international standards.")
    PHYSICAL_10_TO_20M = (4, "Divided. Defined median 10 m to under 20 m wide.")
    PHYSICAL_20M_PLUS = (3, "Divided. Defined median 20 m wide or more.")
    ONE_WAY = (13, "One-way road: no opposing flow.")


class PavedShoulder(int, DocumentedEnum):
    """Width of the paved shoulder, from the edge line to the edge of paving."""

    WIDE = (1, "2.4 m or more, edge line present.")
    MEDIUM = (2, "1 m to under 2.4 m, edge line present.")
    NARROW = (3, "0 m to under 1 m, edge line present.")
    NONE = (4, "No paved shoulder: no edge line.")


class Lanes(int, DocumentedEnum):
    """Traffic lanes in the direction of travel."""

    FOUR_PLUS = (4, "Four or more lanes in the direction of travel.")
    THREE = (3, "Three lanes in the direction of travel.")
    THREE_AND_TWO = (6, "Undivided only. Three lanes one way, two the other.")
    TWO = (2, "Two lanes in the direction of travel.")
    TWO_AND_ONE = (5, "Undivided only. Two lanes one way, one the other.")
    ONE = (1, "One lane in the direction of travel.")


class LaneWidth(int, DocumentedEnum):
    """The narrowest lane's width, centre of edge line to centre of adjacent marking."""

    WIDE = (1, "3.25 m or more.")
    MEDIUM = (2, "2.75 m to under 3.25 m.")
    NARROW = (3, "Under 2.75 m.")


class Curvature(int, DocumentedEnum):
    """Horizontal alignment, by approximate curve radius and safe approach speed."""

    STRAIGHT = (1, "Straight or gently curving.")
    MODERATE = (2, "Moderate curve.")
    SHARP = (3, "Sharp curve.")
    VERY_SHARP = (4, "Very sharp curve.")


class Grade(int, DocumentedEnum):
    """Gradient along the road, up or down."""

    FLAT_OR_GENTLE = (1, "0% to under 7.5%: under 7.5 m rise per 100 m.")
    MODERATE = (4, "7.5% to under 10%.")
    STEEP = (5, "10% or more.")

**Roadside severity.** The most structured attribute: two enums, one for how far the
hazard is from the edge line, one for what it is. A record pairs them for each side
(section 5). The object enum lists hazards from the most to the least dangerous,
with barriers coded first only when they fully protect.

In [ ]:
class RoadsideDistance(int, DocumentedEnum):
    """Distance from the edge line of the nearest driving lane to the roadside object."""

    UNDER_1M = (1, "0 m to under 1 m.")
    FROM_1_TO_5M = (2, "1 m to under 5 m.")
    FROM_5_TO_10M = (3, "5 m to under 10 m.")
    OVER_10M = (4, "10 m or more. Also used when there is no object.")


class RoadsideObject(int, DocumentedEnum):
    """The roadside object with the highest risk. Listed highest to lowest."""

    CLIFF = (10, "Cliff or equivalent hazard, such as the edge of a high bridge: certain fatality at any speed.")
    TREE = (11, "Tree trunk over 10 cm in diameter.")
    RIGID_POLE = (12, "Rigid sign, post or pole over 10 cm in diameter, such as a lighting column.")
    UNPROTECTED_BARRIER_END = (15, "Aggressive safety barrier end. Also used for damaged barrier sections.")
    AGGRESSIVE_VERTICAL_FACE = (5, "Irregular rock face, wall or non-standard barrier.")
    UPWARDS_SLOPE_ROLLOVER = (6, "Cut face at least 2 m high that is likely to roll a vehicle (15 to 75 degrees).")
    DEEP_DITCH = (8, "Ditch or culvert more than 1 m deep.")
    DOWNWARDS_SLOPE = (9, "Slope down from the road steeper than 1:4 with a drop over 1 m.")
    LOW_RIGID_OBJECT = (16, "Low rigid object at least 20 cm high: boulders, non-standard curbs, low walls.")
    RIGID_STRUCTURE = (13, "Solid structure that decelerates a vehicle rapidly: bridge supports, culvert headwalls.")
    UPWARDS_SLOPE_NO_ROLLOVER = (7, "Cut face at least 2 m high at 75 degrees or steeper that a vehicle slides along.")
    SEMI_RIGID_STRUCTURE = (14, "Structure that deforms slightly when hit: houses, bus shelters, masonry fences.")
    BARRIER_CONCRETE = (2, "Continuous, undamaged concrete safety barrier.")
    BARRIER_METAL = (1, "Continuous, undamaged metal safety barrier (not wire rope).")
    BARRIER_WIRE_ROPE = (4, "Continuous, undamaged wire rope safety barrier.")
    BARRIER_MOTORCYCLE_FRIENDLY = (3, "Continuous, undamaged barrier designed to restrain motorcycles.")
    NO_OBJECT = (17, "No object within 20 m of the road.")

**Intersections, property access, parking, and pedestrians.**

In [ ]:
class IntersectionType(int, DocumentedEnum):
    """Presence and type of intersection. Listed highest to lowest risk."""

    FOUR_LEG = (8, "Four or more legs, no signals, no protected turn lane.")
    FOUR_LEG_PROTECTED_TURN = (7, "Four or more legs with a protected turn lane, no signals.")
    FOUR_LEG_SIGNALISED = (10, "Four or more legs with signals, no protected turn lane.")
    THREE_LEG = (4, "Three legs or T-junction, no signals, no protected turn lane.")
    THREE_LEG_PROTECTED_TURN = (3, "Three legs with a protected turn lane, no signals.")
    MINI_ROUNDABOUT = (17, "Roundabout under 4 m radius, typically in low-speed urban areas.")
    THREE_LEG_SIGNALISED = (6, "Three legs with signals, no protected turn lane.")
    FOUR_LEG_SIGNALISED_PROTECTED_TURN = (9, "Four or more legs with signals and a protected turn lane.")
    THREE_LEG_SIGNALISED_PROTECTED_TURN = (5, "Three legs with signals and a protected turn lane.")
    ROUNDABOUT = (2, "Circular intersection around a central island of 4 m radius or more.")
    RAILWAY_PASSIVE = (13, "At-grade railway crossing with give way or stop signage only, or none.")
    MERGE_LANE = (1, "Merge lane for traffic entering the road.")
    RAILWAY_ACTIVE = (14, "At-grade railway crossing with flashing lights or boom gates.")
    MEDIAN_CROSSING_INFORMAL = (15, "Informal U-turn location in the median, without acceleration or deceleration lanes.")
    MEDIAN_CROSSING_FORMAL = (16, "Formal, designed U-turn location in the median.")
    NONE = (12, "No intersection in this bin.")


class PropertyAccess(int, DocumentedEnum):
    """Driveways and minor access lanes in the bin. Listed highest to lowest risk."""

    COMMERCIAL = (1, "One or more commercial access points: service stations, shops, small car parks.")
    RESIDENTIAL_3_PLUS = (2, "Three or more residential access points, or one serving three or more properties.")
    RESIDENTIAL_UNDER_3 = (3, "Fewer than three residential access points, including minor farm tracks.")
    NONE = (4, "No residential or commercial access points.")


class VehicleParking(int, DocumentedEnum):
    """Parking within 2 m of the outside edge of the driveable lane."""

    TWO_SIDES = (3, "Vehicles park on both sides.")
    ONE_SIDE = (2, "Vehicles park on one side.")
    NONE = (1, "No parking.")


class PedestrianCrossing(int, DocumentedEnum):
    """Purpose-built crossing facility. Listed highest to lowest risk."""

    NO_FACILITY = (7, "No provision for pedestrians crossing.")
    REFUGE_ONLY = (6, "A central refuge island, without a marked crossing.")
    MARKED = (5, "Marked crossing, no refuge island, not signalised.")
    RAISED_UNMARKED = (17, "Unmarked crossing on a raised platform, no refuge.")
    MARKED_WITH_REFUGE = (4, "Marked crossing with a refuge island.")
    RAISED_UNMARKED_WITH_REFUGE = (16, "Unmarked crossing on a raised platform, with a refuge.")
    RAISED_MARKED = (15, "Marked crossing on a raised platform, no refuge.")
    RAISED_MARKED_WITH_REFUGE = (14, "Marked crossing on a raised platform, with a refuge.")
    SIGNALISED = (3, "Crossing with pedestrian-specific signals that give a pedestrian phase.")
    SIGNALISED_WITH_REFUGE = (2, "Signalised crossing with a refuge island.")
    GRADE_SEPARATED = (1, "Overbridge or underpass: no conflict with traffic.")


class Sidewalk(int, DocumentedEnum):
    """Pedestrian path along one side, by distance from the nearest driving lane."""

    NONE = (5, "No usable sidewalk, or pedestrians are forced onto the roadway.")
    INFORMAL_UNDER_1M = (7, "Informal path only, under 1 m from the nearest driving lane.")
    INFORMAL_1M_PLUS = (6, "Informal path only, 1 m or more from the nearest driving lane.")
    UNDER_1M = (4, "Sidewalk under 1 m from the nearest driving lane, no barrier.")
    FROM_1_TO_3M = (3, "Sidewalk 1 m to 3 m from the nearest driving lane, no barrier.")
    FROM_3M = (2, "Sidewalk 3 m or more from the nearest driving lane, no barrier.")
    PHYSICAL_BARRIER = (1, "Sidewalk separated from traffic by a barrier that restrains a vehicle at the posted speed.")


class BicycleFacility(int, DocumentedEnum):
    """Purpose-built bicycle facility present for the whole bin."""

    NONE = (4, "No provision for bicycles, or only facilities of poor standard.")
    SIGNED_SHARED_ROADWAY = (6, "No specific provision, but signed as a preferred bicycle route.")
    EXTRA_WIDE_OUTSIDE_LANE = (5, "Outermost lane 4.2 m or wider.")
    BICYCLE_LANE = (3, "Dedicated on-road lane separated by markings or under 1 m of raised or paved surface.")
    SHARED_USE_PATH = (7, "Path shared with pedestrians, separated from traffic by 1 m or more.")
    SEGREGATED_PATH = (2, "Dedicated path separated from traffic by 1 m or more of raised or paved surface.")
    SEGREGATED_PATH_WITH_BARRIER = (1, "Dedicated path behind a physical barrier or 10 m or more from the road.")


class Roadworks(int, DocumentedEnum):
    """Major construction in progress."""

    NONE = (1, "No construction or road works.")
    MINOR = (2, "Works present, but every attribute could still be coded accurately.")
    MAJOR = (3, "Road significantly changed; attributes cannot be coded. No star rating is generated.")


class SchoolZone(int, DocumentedEnum):
    """School zone warning, from start to end of the zone."""

    FLASHING_BEACONS = (1, "School zone with flashing beacons.")
    STATIC_SIGNS = (2, "School zone with static signs or road markings.")
    NO_WARNING = (3, "School present, no zone warning signs or markings.")
    NOT_APPLICABLE = (4, "No school at the location.")

Every member of every enum carries a description. Check that none was left blank:
a `DocumentedEnum` member without a doc is valid Python and a hole in the schema.

In [ ]:
ENUMS = [
    Carriageway, Presence, Adequacy, AreaType, LandUse, SpeedLimit, MedianType, PavedShoulder, Lanes,
    LaneWidth, Curvature, Grade, RoadsideDistance, RoadsideObject, IntersectionType, PropertyAccess,
    VehicleParking, PedestrianCrossing, Sidewalk, BicycleFacility, Roadworks, SchoolZone,
]
undocumented = [f"{e.__name__}.{m.name}" for e in ENUMS for m in e if not m.__doc__]
print(f"{len(ENUMS)} enums, {sum(len(e) for e in ENUMS)} values, undocumented: {undocumented}")

**Try it:** two columns are not modelled yet. `Differential speed limits` holds `1` in
every row and `Facilities for motorised two wheelers` holds `6` in every row. Per
the manual: differential speed is `1` not present, `2` present (a speed gap over
20 km/h between cars and trucks or motorcycles); motorcycle facilities run from `1`
(two-way path with barrier) to `6` (none). Write a `DocumentedEnum` for each, using
the manual's codes. Should a column that never varies in this file be in the schema?

## 5. The record

Three structs and one model. A struct groups fields that belong together and, like
`Survey` in notebook 5, rejects keys nobody declared.

`RoadsideSeverity` pairs a distance with an object. It is used twice, once per side,
which is the reason it is a struct rather than four loose fields.

In [ ]:
from datetime import date
from typing import Annotated, NewType

from pydantic import BaseModel, Field

from overture.schema.system.feature import Feature
from overture.schema.system.geometric import Geometry, GeometryType, GeometryTypeConstraint
from overture.schema.system.model_constraint import FieldEqCondition, forbid_if, no_extra_fields
from overture.schema.system.numeric import uint8, uint32

StarRating = NewType(
    "StarRating",
    Annotated[uint8, Field(ge=1, le=5, description="Star rating from 1 (least safe) to 5 (safest).")],
)


@no_extra_fields
class RoadsideSeverity(BaseModel):
    """The most hazardous roadside object on one side of the road, and how far away it is."""

    distance: RoadsideDistance
    object: RoadsideObject


@no_extra_fields
class VehicleStarRating(BaseModel):
    """Star ratings for vehicle occupants."""

    car: Annotated[StarRating, Field(description="Smoothed over neighbouring bins. The published headline rating.")]
    car_raw: Annotated[StarRating, Field(description="The bin's own rating, before smoothing.")]


Score = Annotated[float, Field(ge=0)]


@no_extra_fields
class VehicleScores(BaseModel):
    """Star Rating Scores for vehicle occupants. A higher score is a higher risk of death or serious injury."""

    run_off_driver_side: Annotated[Score, Field(description="Run-off-road, loss of control, driver side.")]
    run_off_passenger_side: Annotated[Score, Field(description="Run-off-road, loss of control, passenger side.")]
    head_on: Annotated[Score, Field(description="Head-on, loss of control.")]
    head_on_overtaking: Annotated[Score, Field(description="Head-on while overtaking.")]
    intersection: Annotated[Score, Field(description="Intersection crashes.")]
    property_access: Annotated[Score, Field(description="Property access crashes.")]
    total: Score
    total_smoothed: Score

The model. Its fields mirror the columns, with names in snake_case. Compare two
decisions to notebook 5's `RoadSafetyRating`:

- The star rating and scores are **optional**, and a rule ties that to the data:
  where roadworks are major, no rating is generated, and a rating that appears
  anyway is an error. Section 2 showed the data obeys this. The rule is written with
  `@forbid_if`, as data, so it reaches the JSON Schema (section 7).
- `location_id` is a field, not the feature's `id`. `Feature.id` is the slot for a
  stable identifier, and `Location ID` is not one: it names a row in one file.

In [ ]:
@no_extra_fields
@forbid_if(["star_rating", "scores"], FieldEqCondition("roadworks", Roadworks.MAJOR))
class AusrapSegment(Feature):
    """One 100 m bin of one carriageway, assessed for road safety with the iRAP methodology."""

    geometry: Annotated[
        Geometry,
        GeometryTypeConstraint(GeometryType.LINE_STRING),
        Field(description="Straight line from the start to the end of the 100 m bin."),
    ]
    location_id: Annotated[int, Field(description="This file's identifier for the bin. Not stable across surveys.")]
    smoothed_section_id: Annotated[int, Field(description="The smoothed section the bin belongs to.")]
    carriageway: Carriageway
    surveyed_on: Annotated[date | None, Field(description="When the road was filmed.")] = None

    # Context
    area_type: AreaType
    land_use_driver_side: LandUse
    land_use_passenger_side: LandUse
    roadworks: Roadworks
    school_zone_warning: SchoolZone
    aadt: Annotated[uint32, Field(description="Annual average daily traffic, vehicles per day.")]

    # Speed
    speed_limit: SpeedLimit
    motorcycle_speed_limit: SpeedLimit
    truck_speed_limit: SpeedLimit
    traffic_calming: Annotated[Presence, Field(description="Features that reduce operating speed 5 to 10 km/h below the limit.")]

    # Cross-section
    median_type: MedianType
    centreline_rumble_strips: Presence
    shoulder_rumble_strips: Presence
    paved_shoulder_driver_side: PavedShoulder
    paved_shoulder_passenger_side: PavedShoulder
    number_of_lanes: Lanes
    lane_width: LaneWidth
    service_road: Presence

    # Alignment and surface
    curvature: Curvature
    grade: Grade
    delineation: Adequacy
    street_lighting: Presence
    vehicle_parking: VehicleParking

    # Roadside
    roadside_driver_side: RoadsideSeverity
    roadside_passenger_side: RoadsideSeverity

    # Intersections and access
    intersection_type: IntersectionType
    intersection_channelisation: Presence
    property_access_points: PropertyAccess

    # Pedestrians and bicycles
    pedestrian_crossing_inspected_road: PedestrianCrossing
    pedestrian_crossing_intersecting_road: PedestrianCrossing
    pedestrian_fencing: Presence
    sidewalk_driver_side: Sidewalk
    sidewalk_passenger_side: Sidewalk
    bicycle_facilities: BicycleFacility

    # Results
    star_rating: VehicleStarRating | None = None
    scores: VehicleScores | None = None

## 6. Check the schema against the data

A schema that has never met the data is a guess. Two checks, in order of cost.

**Every code in the file is an enum member.** This is a set comparison in SQL over
all 245,128 rows: for each modelled column, the codes the data holds minus the
codes the enum defines. An empty difference means the enum covers the file. The
second column is the reverse: values the manual defines that this file never uses.

In [ ]:
import pandas as pd

# CSV column -> enum. Columns that share an enum are listed once per column.
CODED = {
    "Carriageway": Carriageway, "Area type": AreaType,
    "Land use - driver-side": LandUse, "Land use - passenger-side": LandUse,
    "Speed limit": SpeedLimit, "Motorcycle speed limit": SpeedLimit, "Truck speed limit": SpeedLimit,
    "Speed management / traffic calming": Presence, "Median type": MedianType,
    "Centreline rumble strips": Presence, "Shoulder rumble strips": Presence,
    "Roadside severity - driver-side distance": RoadsideDistance,
    "Roadside severity - driver-side object": RoadsideObject,
    "Roadside severity - passenger-side distance": RoadsideDistance,
    "Roadside severity - passenger-side object": RoadsideObject,
    "Paved shoulder - driver-side": PavedShoulder, "Paved shoulder - passenger-side": PavedShoulder,
    "Intersection type": IntersectionType, "Intersection channelisation": Presence,
    "Property access points": PropertyAccess, "Number of lanes": Lanes, "Lane width": LaneWidth,
    "Curvature": Curvature, "Grade": Grade, "Delineation": Adequacy, "Street lighting": Presence,
    "Pedestrian crossing facilities - inspected road": PedestrianCrossing,
    "Pedestrian crossing facilities - intersecting road": PedestrianCrossing,
    "Pedestrian fencing": Presence, "Vehicle parking": VehicleParking,
    "Sidewalk - driver-side": Sidewalk, "Sidewalk - passenger-side": Sidewalk,
    "Service road": Presence, "Facilities for bicycles": BicycleFacility,
    "Roadworks": Roadworks, "School zone warning": SchoolZone,
}

rows = []
for column, enum in CODED.items():
    defined = {m.value for m in enum}
    in_data = {r[0] for r in con.execute(f'SELECT DISTINCT "{column}" FROM ausrap').fetchall()}
    rows.append({
        "column": column, "enum": enum.__name__,
        "in_data_not_enum": sorted(in_data - defined),
        "in_enum_not_data": sorted(defined - in_data),
    })
coverage = pd.DataFrame(rows)
print(f"{len(coverage)} coded columns; codes in the data that no enum defines: {sum(len(u) for u in coverage.in_data_not_enum)}")
print("Columns where the manual defines codes this file never uses:")
print(coverage[coverage.in_enum_not_data.str.len() > 0][["column", "in_enum_not_data"]].to_string(index=False))

No column holds a code outside its enum. The table lists the part of iRAP's
vocabulary that this file never uses (speed limits above 110 km/h, motorcycle-path
carriageways), which is the argument for modelling the manual rather than the file.

That check does not exercise the control: if it could not fail, "no unknown codes"
would mean nothing. Remove one defined value and it reports the gap.

In [ ]:
in_data = {r[0] for r in con.execute('SELECT DISTINCT "Median type" FROM ausrap').fetchall()}
without_wire_rope = {m.value for m in MedianType if m is not MedianType.BARRIER_WIRE_ROPE}
sorted(in_data - without_wire_rope)

**Every row in an area validates against the model.** Build a record per row the way
a data pipeline would: rename columns, stitch the start and end coordinates into a
`LineString`, group the roadside and score columns into their structs. Central
Melbourne is a few thousand bins.

In [ ]:
AOI = (144.90, -37.86, 145.02, -37.78)  # xmin, ymin, xmax, ymax
xmin, ymin, xmax, ymax = AOI

records = con.execute(f"""
    SELECT
        'LINESTRING (' || Longitude || ' ' || Latitude || ', ' || "End Longitude" || ' ' || "End Latitude" || ')' AS geometry,
        "Location ID" AS location_id,
        "Smoothed Section ID" AS smoothed_section_id,
        Carriageway AS carriageway,
        try_strptime(split_part("Road survey date", ' ', 1), '%d/%m/%Y')::DATE AS surveyed_on,
        "Area type" AS area_type,
        "Land use - driver-side" AS land_use_driver_side,
        "Land use - passenger-side" AS land_use_passenger_side,
        Roadworks AS roadworks,
        "School zone warning" AS school_zone_warning,
        "Vehicle flow (AADT)" AS aadt,
        "Speed limit" AS speed_limit,
        "Motorcycle speed limit" AS motorcycle_speed_limit,
        "Truck speed limit" AS truck_speed_limit,
        "Speed management / traffic calming" AS traffic_calming,
        "Median type" AS median_type,
        "Centreline rumble strips" AS centreline_rumble_strips,
        "Shoulder rumble strips" AS shoulder_rumble_strips,
        "Paved shoulder - driver-side" AS paved_shoulder_driver_side,
        "Paved shoulder - passenger-side" AS paved_shoulder_passenger_side,
        "Number of lanes" AS number_of_lanes,
        "Lane width" AS lane_width,
        "Service road" AS service_road,
        Curvature AS curvature,
        Grade AS grade,
        Delineation AS delineation,
        "Street lighting" AS street_lighting,
        "Vehicle parking" AS vehicle_parking,
        {{'distance': "Roadside severity - driver-side distance",
          'object': "Roadside severity - driver-side object"}} AS roadside_driver_side,
        {{'distance': "Roadside severity - passenger-side distance",
          'object': "Roadside severity - passenger-side object"}} AS roadside_passenger_side,
        "Intersection type" AS intersection_type,
        "Intersection channelisation" AS intersection_channelisation,
        "Property access points" AS property_access_points,
        "Pedestrian crossing facilities - inspected road" AS pedestrian_crossing_inspected_road,
        "Pedestrian crossing facilities - intersecting road" AS pedestrian_crossing_intersecting_road,
        "Pedestrian fencing" AS pedestrian_fencing,
        "Sidewalk - driver-side" AS sidewalk_driver_side,
        "Sidewalk - passenger-side" AS sidewalk_passenger_side,
        "Facilities for bicycles" AS bicycle_facilities,
        CASE WHEN "Vehicle Star Rating Smoothed" IS NULL THEN NULL
             ELSE {{'car': "Vehicle Star Rating Smoothed", 'car_raw': "Vehicle Star Rating Raw"}} END AS star_rating,
        CASE WHEN "Vehicle SRS Total" IS NULL THEN NULL
             ELSE {{'run_off_driver_side': "Vehicle SRS Run-Off LOC Driver-Side",
                    'run_off_passenger_side': "Vehicle SRS Run-Off LOC Passenger-Side",
                    'head_on': "Vehicle SRS Head-On LOC",
                    'head_on_overtaking': "Vehicle SRS Head-On Overtaking",
                    'intersection': "Vehicle SRS Intersection",
                    'property_access': "Vehicle SRS Property Access",
                    'total': "Vehicle SRS Total",
                    'total_smoothed': "Vehicle SRS Total Smoothed"}} END AS scores
    FROM ausrap
    WHERE Longitude BETWEEN {xmin} AND {xmax} AND Latitude BETWEEN {ymin} AND {ymax}
""").fetchdf().to_dict("records")

segments = [AusrapSegment.model_validate(r) for r in records]
print(f"{len(segments):,} bins validated")

One validated record, as JSON. `exclude_none` drops what the record does not have,
and the codes appear as the integers the data holds.

In [ ]:
import json

print(json.dumps(segments[0].model_dump(mode="json", exclude_none=True), indent=2, default=str)[:1800])

The rule fires on real data too. Take a valid row and set the roadworks to major
while keeping its star rating:

In [ ]:
AusrapSegment.model_validate({**records[0], "roadworks": Roadworks.MAJOR})

And a code the manual does not define fails with the enum's allowed values:

In [ ]:
AusrapSegment.model_validate({**records[0], "lane_width": 4})

## 7. Emit the schema

`json_schema` turns the model into JSON Schema: the artifact other tools and other
languages read. Write it to `results/` and see what survived the trip.

In [ ]:
from overture.schema.system.json_schema import json_schema

schema = json_schema(AusrapSegment)
RESULTS.mkdir(exist_ok=True)
(RESULTS / "ausrap_segment.schema.json").write_text(json.dumps(schema, indent=2))

props = schema["properties"]["properties"]
print("required:", props["required"][:6], "...")
print("fields:", len(props["properties"]), " definitions:", len(schema["$defs"]))
print("rule:", json.dumps(props["if"]), "->", json.dumps(props["then"])[:90])

The rule is in the schema as an `if`/`then`, so a validator in any language
enforces it. The enum's *values* are there too:

In [ ]:
schema["$defs"]["LaneWidth"]

The values (`1, 2, 3`) are in the JSON Schema; their meanings are not. For an
integer enum the schema stops at `{"enum": [1, 2, 3]}`, and the descriptions you
wrote live on the Python members. A consumer of the JSON Schema alone gets codes
with no vocabulary. Closing that gap is the strongest argument for the string-valued
design (`"narrow"`), and it is where you decide what the schema is for.

**Try it:** redefine `LaneWidth` with string values (`"wide"`, `"medium"`, `"narrow"`)
and look at `json_schema(LaneWidth)`. Then change the SQL that builds `records` so the
data validates again. Where does the code-to-string mapping live, and who owns it?

**Try it:** the manual says that where `intersection_type` is `NONE`, the intersection
channelisation must be `NOT_PRESENT`. The data agrees on every bin with no
intersection. Can you write that with `@require_if` or `@forbid_if`? What would
the tools need to support to state it?

## What you have

A model for one AusRAP record, a documented enum for every coded column (the
manual's meaning for every value) and a rule, validated against the real file and
emitted as JSON Schema. What is verified against the data is stated in section 2; the
rest of the vocabulary is the manual's, at version 5.3.

Registering it with the schema tools (the entry point and `uv sync --all-packages`
step from the [`my-schema` README](../my-schema/README.md)) would let you generate
documentation pages for it with `overture-codegen`.